# MatrixLLM — Local Deployment Demo (Enterprise-Style)

This notebook is a **hands-on, copy/paste-ready** tutorial for using a **local MatrixLLM gateway** (OpenAI-compatible API) with clean, production-grade examples.

**What you'll do**
- Start the gateway locally with **auto-reload**
- Verify the service (health + model readiness)
- Call **Chat Completions** with the official `openai` Python SDK
- Use **streaming** responses
- Apply **enterprise hygiene**: configuration, secrets, timeouts, retries, and logging

## Multi-Provider Routing
MatrixLLM supports routing to multiple LLM providers:
- **OpenAI** (`openai/gpt-4o-mini`, `openai/gpt-4o`)
- **Anthropic** (`anthropic/claude-3-5-sonnet`, `anthropic/claude-3-haiku`)
- **Google Gemini** (`google/gemini-pro`, `google/gemini-flash`)
- **IBM watsonx.ai** (`ibm/granite-13b-chat-v2`)
- **Local Ollama** (`deepseek-r1`, `llama3`, etc.)
- **MatrixNode** (distributed relay nodes)

> **Assumptions**
> - You already started MatrixLLM (or you will in Step 1) and see output similar to:
>
> `... start --host 0.0.0.0 --port 11435 --reload ...`
>
> with:
> - Local API: `http://localhost:11435/v1`
> - Health: `http://localhost:11435/health`
> - Key: `sk-matrixllm-...`

**Last updated:** 2026-01-22

## 0) Prerequisites

### Required
- Python **3.10+**
- A running **MatrixLLM** gateway (local)
- A model available in your gateway (example below uses `deepseek-r1`)

### Install client dependencies (this notebook)

In [ ]:
# If you're running this notebook locally:
# python -m pip install -U openai requests python-dotenv

!python -m pip -q install -U openai requests python-dotenv

## 1) Start the gateway (auto-reload)

Run this command **in a terminal** (not inside the notebook) from your MatrixLLM repo/venv:

```bash
.venv/bin/python3 -m matrixllm.cli.main start --host 0.0.0.0 --port 11435 --reload --log-level info
```

Or simply:
```bash
matrixllm start --host 0.0.0.0 --port 11435 --reload --log-level info
```

When it's ready you should see a banner like:

- Local API: `http://localhost:11435/v1`
- Health: `http://localhost:11435/health`
- Key: `sk-matrixllm-...`

### Security note (enterprise)
Treat the key like a secret:
- do **not** commit it to git
- prefer environment variables (or a secrets manager)

## 2) Configure the client (base URL + API key)

You can provide the key in either header style:
- `X-API-Key: sk-...`
- `Authorization: Bearer sk-...`

The `openai` Python SDK uses `Authorization: Bearer ...` automatically.

In [ ]:
import os
from dotenv import load_dotenv

# Load variables from .env file into the environment
# This defaults to looking for a file named ".env" in the current directory
load_dotenv() 

# Now os.getenv will find the values defined in your .env file
MATRIXLLM_BASE_URL = os.getenv("MATRIXLLM_BASE_URL", "http://localhost:11435/v1")
MATRIXLLM_API_KEY  = os.getenv("MATRIXLLM_API_KEY", "sk-matrixllm-REPLACE_ME")

print("Base URL:", MATRIXLLM_BASE_URL)
# Check if key is loaded (and not the default placeholder)
print("API key set:", MATRIXLLM_API_KEY.startswith("sk-matrixllm-") and "REPLACE_ME" not in MATRIXLLM_API_KEY)

In [ ]:
import requests
from urllib.parse import urlparse

def gateway_health(base_url: str, timeout_s: int = 10):
    """Return (ok: bool, payload_or_error: object)."""
    try:
        health_url = base_url.replace("/v1", "") + "/health"
        r = requests.get(health_url, timeout=timeout_s)
        r.raise_for_status()
        return True, r.json()
    except Exception as e:
        return False, str(e)

def require_gateway(base_url: str):
    ok, info = gateway_health(base_url, timeout_s=10)
    if ok:
        print("✅ Gateway reachable:", base_url)
        print("Health:", info)
        return True
    print("⚠️ Gateway NOT reachable:", base_url)
    print("Reason:", info)
    print("\nTip: start MatrixLLM first, or set the correct base URL / tunnel URL.")
    return False

## 3) Health check (fast validation)

This should return HTTP 200 and a simple JSON payload.

In [ ]:
# Health check (safe): won't crash the notebook if the gateway isn't up yet.
health_url = MATRIXLLM_BASE_URL.replace("/v1", "") + "/health"
ok, info = gateway_health(MATRIXLLM_BASE_URL, timeout_s=20)
if ok:
    info
else:
    print("Skipping health check because gateway is not reachable.")
    print("Reason:", info)

## 4) List models (optional)

If your gateway supports the standard OpenAI-compatible endpoint, this will show the models it exposes.

With multi-provider routing enabled, you'll see models from all configured providers (e.g., `openai/gpt-4o-mini`, `anthropic/claude-3-5-sonnet`, `deepseek-r1`).

In [ ]:
if not require_gateway(MATRIXLLM_BASE_URL):
    print('⏭️ Skipping this cell (gateway not reachable).')
else:
    from openai import OpenAI

    client = OpenAI(
        base_url=MATRIXLLM_BASE_URL,
        api_key=MATRIXLLM_API_KEY,
    )

    # Some gateways support /models; if yours doesn't, skip this cell.
    try:
        models = client.models.list()
        [m.id for m in models.data][:10]
    except Exception as e:
        print("Model listing not available (this is OK). Error:", e)

## 5) Chat Completions (basic)

This is the core OpenAI-compatible usage.

### Multi-Provider Examples
```python
# Local Ollama model
model_name = "deepseek-r1"

# OpenAI (requires OPENAI_COMPAT_API_KEY)
model_name = "openai/gpt-4o-mini"

# Anthropic (requires ANTHROPIC_API_KEY)
model_name = "anthropic/claude-3-5-sonnet"

# Google Gemini (requires GEMINI_API_KEY)
model_name = "google/gemini-pro"

# IBM watsonx (requires WATSONX_API_KEY)
model_name = "ibm/granite-13b-chat-v2"
```

In [ ]:
if not require_gateway(MATRIXLLM_BASE_URL):
    print('⏭️ Skipping this cell (gateway not reachable).')
else:
    model_name = os.getenv("MATRIXLLM_MODEL", "deepseek-r1")

    resp = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a concise assistant."},
            {"role": "user", "content": "Give me 3 bullet points on why gateways are useful in enterprise AI deployments."},
        ],
        temperature=0.2,
    )

    print(resp.choices[0].message.content)

## 6) Streaming responses (production-friendly UX)

Streaming is ideal for:
- chat UIs
- long outputs
- faster perceived latency

In [ ]:
if not require_gateway(MATRIXLLM_BASE_URL):
    print('⏭️ Skipping this cell (gateway not reachable).')
else:
    stream = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "Answer in a short paragraph."},
            {"role": "user", "content": "Explain what auto-reload means for a local API gateway in development."},
        ],
        temperature=0.2,
        stream=True,
    )

    out = []
    for event in stream:
        delta = event.choices[0].delta
        if delta and delta.content:
            out.append(delta.content)
            print(delta.content, end="", flush=True)

    print()

## 7) Robust client wrapper (timeouts, retries, structured errors)

Below is a small helper you can copy into production services.

**Why?**
- predictable timeouts
- transparent retry policy
- consistent logging and error handling

In [ ]:
if not require_gateway(MATRIXLLM_BASE_URL):
    print('⏭️ Skipping this cell (gateway not reachable).')
else:
    from dataclasses import dataclass
    from typing import List, Dict, Optional
    import time
    import logging

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("matrixllm-demo")

    @dataclass
    class ChatConfig:
        model: str = "deepseek-r1"
        temperature: float = 0.2
        max_retries: int = 2
        request_timeout_s: int = 60  # SDK uses httpx; timeouts are handled internally

    def chat_once(messages: List[Dict[str, str]], cfg: ChatConfig) -> str:
        last_err: Optional[Exception] = None
        for attempt in range(cfg.max_retries + 1):
            try:
                t0 = time.time()
                resp = client.chat.completions.create(
                    model=cfg.model,
                    messages=messages,
                    temperature=cfg.temperature,
                )
                dt = time.time() - t0
                logger.info("chat completion ok (%.2fs)", dt)
                return resp.choices[0].message.content
            except Exception as e:
                last_err = e
                logger.warning("attempt %d failed: %s", attempt + 1, e)
                if attempt < cfg.max_retries:
                    time.sleep(1.5 * (attempt + 1))
        raise RuntimeError(f"All retries exhausted. Last error: {last_err}") from last_err

    print(
        chat_once(
            [{"role":"user","content":"Write a one-sentence definition of an API gateway."}],
            ChatConfig(model=model_name),
        )
    )

## 8) Multi-Provider Routing Demo

This section demonstrates how to call different providers through the same MatrixLLM gateway.

**Prerequisites:**
- Configure provider API keys in your `.env` file
- Set `ROUTING_MODE=prefix` in your configuration

In [ ]:
if not require_gateway(MATRIXLLM_BASE_URL):
    print('⏭️ Skipping this cell (gateway not reachable).')
else:
    # Test different providers (uncomment the ones you have configured)
    providers_to_test = [
        # ("openai/gpt-4o-mini", "Say hello in one sentence."),
        # ("anthropic/claude-3-haiku", "What is 2+2? Answer briefly."),
        # ("google/gemini-pro", "Name one planet."),
        ("deepseek-r1", "Say hello."),  # Local Ollama (always available)
    ]
    
    for model, prompt in providers_to_test:
        try:
            print(f"\n--- Testing: {model} ---")
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=50,
            )
            print(f"Response: {resp.choices[0].message.content[:100]}")
        except Exception as e:
            print(f"Error: {e}")

## 9) Troubleshooting checklist

### Gateway not reachable
- Confirm the process is running:
  - `Uvicorn running on http://0.0.0.0:11435`
- Verify the URL:
  - Base: `http://localhost:11435/v1`
  - Health: `http://localhost:11435/health`

### 401 / unauthorized
- Ensure you are using the correct key from the banner:
  - `Authorization: Bearer sk-matrixllm-...`

### Model not found
- Verify the model name in your gateway banner, and set:
  - `MATRIXLLM_MODEL=...`
- For multi-provider models, ensure the provider is configured:
  - OpenAI: `OPENAI_COMPAT_API_KEY`
  - Anthropic: `ANTHROPIC_API_KEY`
  - Gemini: `GEMINI_API_KEY`
  - watsonx: `WATSONX_API_KEY`

### Dev workflow (auto-reload)
- With `--reload`, edits to the code trigger an automatic server reload.
  - Great for rapid iteration
  - Avoid in production (use a fixed build + stable config)

## Next steps

- Add observability: request IDs, structured logs, metrics.
- Use a reverse proxy (TLS termination) for shared environments.
- Rotate keys and enforce allowlists/rate limits.
- Configure multi-provider routing with your preferred LLM providers.

If you want, I can also generate:
- a small **FastAPI** client service that calls your gateway
- a **Docker Compose** demo for local dev + reverse proxy + metrics